# 🚀 Google Colab GPU Training Bridge

Since the Google Colab file upload widget (`files.upload()`) is disabled inside VS Code's Jupyter interface, use **Google Drive** to sync your files:

### 📂 Setup Instructions:
1. Run this local terminal command in your workspace to package the project:
   `zip -r Stock_Transformer.zip src/ main.py predict.py app.py requirements.txt -x "stock_env/*"` 
2. Go to your [Google Drive](https://drive.google.com/) browser interface and upload `Stock_Transformer.zip` directly to the root of your Drive.
3. Execute the cells below in order.

In [1]:
# Mount Google Drive (this will prompt you to authorize the mount inside VS Code)
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# Copy and unzip the code
zip_path = "/content/drive/MyDrive/Stock_Transformer.zip"
if os.path.exists(zip_path):
    shutil.copy(zip_path, "/content/Stock_Transformer.zip")
    !unzip -o -q /content/Stock_Transformer.zip -d /content/
    print("✅ Stock_Transformer code successfully unzipped!")
else:
    print(f"❌ Could not find {zip_path} in your Google Drive root. Please upload it first!")

Mounted at /content/drive
✅ Stock_Transformer code successfully unzipped!


In [2]:
# Install dependencies (PyTorch is pre-installed on Colab)
!pip install -q yfinance plotly pandas numpy scikit-learn joblib
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Device: Tesla T4


In [3]:
# Start training on the T4 GPU using a 1e-3 learning rate
!python main.py --epochs 50 --batch_size 256 --lr 1e-3 --d_model 128 --nhead 4 --num_layers 4 --dropout 0.25

2026-06-17 16:09:20.709584: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Preparing global multi-stock dataset for 41 tickers...
  Split: Train ≤ 2024 | Val = 2025 | Test ≥ 2026
Fitting baseline global scalers on AAPL...
  [cache] Fetching fresh AAPL data from yfinance (period=max)...
  [cache] yfinance (direct) failed for AAPL: Yahoo API requires curl_cffi session not <class 'requests.sessions.Session'>. Solution: stop setting session, let YF handle.
  [cache] yfinance returned no data, trying Alpha Vantage fallback...
ALPHA_VANTAGE_KEY not set — cannot use Alpha Vantage fallback.
Traceback (most recent call last):
  File "/content/main.py", line 110, in <module>
    main()
  File "/content/main.py", line 39, in main
    base_input, feature_sc

In [ ]:
# Run model inference to verify (match the d_model and stock_embed_dim you used during training!)
!python predict.py --ticker AAPL --d_model 128 --stock_embed_dim 32

In [ ]:
# Copy trained weights and scalers back to your Google Drive
import os
import shutil

if os.path.exists("best_model.pth"):
    shutil.copy("best_model.pth", "/content/drive/MyDrive/best_model.pth")
    print("✅ Saved best_model.pth to your Google Drive!")

if os.path.exists("models"):
    shutil.make_archive("/content/drive/MyDrive/trained_outputs", "zip", "models")
    print("✅ Saved trained_outputs.zip (scalers) to your Google Drive!")